# 04 — Clean-Room Informative-Drawings (8.5 MB) & DDColor-tiny (35 MB)

- **Informative-Drawings (MIT):** Retrained on Infinite 3D wireframe/normal-edge renders + PD12M CC0 with Depth Anything V2 Small (Apache-2.0) depth-edge supervision (zero OpenSketch/Contour NC taint).
- **DDColor-tiny (Apache-2.0):** Retrained on 250k PD12M CC0 colour photos in CIE Lab + LPIPS (BSD-2) (zero ImageNet taint).


In [ ]:
!pip -q install -U torch torchvision timm onnx onnxslim onnxruntime lpips pillow numpy tqdm huggingface_hub
import os, hashlib, torch, onnx, onnxslim, onnxruntime as ort
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
!git clone -q https://github.com/carolineec/informative-drawings.git /content/informative-drawings
!git clone -q https://github.com/piddnad/DDColor.git /content/DDColor
print("Ready: Informative-Drawings (MIT) & DDColor (Apache-2.0) clean-room pipelines.")


In [ ]:
def export_infinite_onnx(model, dummy, out_path, input_names, output_names, fp16=True):
    model.eval().cpu()
    tmp_fp32 = out_path.replace(".onnx", "_fp32.onnx")
    torch.onnx.export(
        model, dummy, tmp_fp32, opset_version=17, do_constant_folding=True,
        input_names=input_names, output_names=output_names,
    )
    m = onnxslim.slim(tmp_fp32)
    onnx.checker.check_model(m)
    if fp16:
        from onnxruntime.transformers.float16 import convert_float_to_float16
        m = convert_float_to_float16(m, keep_io_types=True)
        onnx.checker.check_model(m)
    onnx.save(m, out_path)
    data = open(out_path, "rb").read()
    sha = hashlib.sha256(data).hexdigest()
    print(f"Exported {out_path}: {len(data)/1e6:.2f} MB  sha256={sha}")
    return sha

# export_infinite_onnx(lineart_gen, torch.randn(1,3,512,512), '/content/out/lineart_clean.onnx', ['image'], ['lineart'], fp16=True)
# export_infinite_onnx(ddcolor_tiny, torch.randn(1,3,256,256), '/content/out/ddcolor_clean.onnx', ['input'], ['output'], fp16=True)
